<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/03_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Calcolo le baseline per tutti i dataset filtrati per N semimosse; Per lo scenario A utilizzo solo la differenza di elo mentre per il B utilizzo tutti i meta-dat (differenza elo assoluti, tempo stimato e validità della partita classificata o meno). Per evitare che i dati non siano influenzati da fluttuazioni casuali uso la cross validation stratificata su 5 fold sul Train set, così estraggo l'accuratezza sotto forma di media e deviazione standard.
Utilizzo poi le metrice ROC-AUC e Log-Loss per una buona calibrazione delle probabilità. Confronto poi il coefficiente teorico dalla formula di Arpad Elo con quelo che risulta empiricamente, dicrepanza giustificata dal fatto che il sistema di rating di Glicko-2 ha misure di incertezza che non ci sono ne modello classico.


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import cross_validate, StratifiedKFold
import warnings

# Disabilita i warning di convergenza per mantenere pulito l'output
warnings.filterwarnings('ignore')

print("1. VALUTAZIONE BASELINE AVANZATA (A: Solo Elo | B: Tutti i Metadati)\n")

soglie = [10, 20, 30, 40]
risultati = []

# Imposta la Cross-Validation a 5 fette (Fold) per calcolare media e deviazione standard
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Inizializza il modello logistico
log_reg = LogisticRegression(random_state=42, max_iter=1000)

for N in soglie:
    # Carica i dataset filtrati per la soglia N
    df_train = pd.read_csv(f'train_features_N{N}.csv')
    df_test = pd.read_csv(f'test_features_N{N}.csv')

    # Definisce il target (1 per vittoria Bianco, 0 per vittoria Nero)
    y_train = df_train['winner']
    y_test = df_test['winner']

    # Calcola la differenza Elo (feature base)
    df_train['rating_diff'] = df_train['white_rating'] - df_train['black_rating']
    df_test['rating_diff'] = df_test['white_rating'] - df_test['black_rating']

    # Definisce i due scenari (A e B)
    features_b = ['rating_diff', 'white_rating', 'black_rating', 'rated', 'tempo_stimato_sec']

    X_train_a = df_train[['rating_diff']]
    X_test_a = df_test[['rating_diff']]

    X_train_b = df_train[features_b]
    X_test_b = df_test[features_b]

    # --- SCENARIO A: Solo Rating Diff ---
    # Valuta la variabilità sul Train Set tramite Cross-Validation
    cv_res_a = cross_validate(log_reg, X_train_a, y_train, cv=cv, scoring='accuracy')
    cv_acc_a_mean = cv_res_a['test_score'].mean() * 100
    cv_acc_a_std = cv_res_a['test_score'].std() * 100

    # Addestra il modello sul Train Set completo e fa le previsioni sul Test Set
    log_reg.fit(X_train_a, y_train)
    y_pred_a = log_reg.predict(X_test_a)
    acc_a = accuracy_score(y_test, y_pred_a)

    # Estrae i pesi matematici solo al primo ciclo (N=10) per l'analisi teorica
    if N == 10:
        coef_elo = log_reg.coef_[0][0]
        intercetta_bianco = log_reg.intercept_[0]

    # --- SCENARIO B: Tutti i Metadati Pre-Partita ---
    # Valuta la variabilità sul Train Set tramite Cross-Validation
    cv_res_b = cross_validate(log_reg, X_train_b, y_train, cv=cv, scoring='accuracy')
    cv_acc_b_mean = cv_res_b['test_score'].mean() * 100
    cv_acc_b_std = cv_res_b['test_score'].std() * 100

    # Addestra il modello e preleva anche le probabilità pure per ROC-AUC e Log-Loss
    log_reg.fit(X_train_b, y_train)
    y_pred_b = log_reg.predict(X_test_b)
    y_prob_b = log_reg.predict_proba(X_test_b)[:, 1]

    acc_b = accuracy_score(y_test, y_pred_b)
    roc_b = roc_auc_score(y_test, y_prob_b)
    loss_b = log_loss(y_test, y_prob_b)

    # Salva la riga riassuntiva per la tabella
    risultati.append({
        'N': N,
        'CV_Train_A (%)': f"{cv_acc_a_mean:.1f} ± {cv_acc_a_std:.2f}",
        'Test_Acc_A (%)': round(acc_a * 100, 2),
        'CV_Train_B (%)': f"{cv_acc_b_mean:.1f} ± {cv_acc_b_std:.2f}",
        'Test_Acc_B (%)': round(acc_b * 100, 2),
        'Test_ROC_AUC_B': round(roc_b, 3),
        'Test_LogLoss_B': round(loss_b, 3)
    })

# Stampa la tabella
print(pd.DataFrame(risultati).to_string(index=False))

print("\n2. ANALISI MATEMATICA: TEORIA ELO VS PRATICA (N=10)")
print(f"Coefficiente Teorico Elo (Formula di Arpad Elo): 0.00576")
print(f"Coefficiente Calcolato dal Modello (rating_diff): {coef_elo:.5f}")
print(f"Intercetta (Vantaggio Innato del Bianco):         {intercetta_bianco:.5f}")

1. VALUTAZIONE BASELINE AVANZATA (A: Solo Elo | B: Tutti i Metadati)

 N CV_Train_A (%)  Test_Acc_A (%) CV_Train_B (%)  Test_Acc_B (%)  Test_ROC_AUC_B  Test_LogLoss_B
10    64.7 ± 0.40           65.15    64.7 ± 0.43           64.98           0.713           0.616
20    64.3 ± 0.66           64.79    64.3 ± 0.71           64.76           0.710           0.619
30    64.0 ± 0.87           64.61    64.0 ± 0.70           64.48           0.705           0.623
40    63.5 ± 0.90           64.08    63.5 ± 0.98           64.20           0.697           0.629

2. ANALISI MATEMATICA: TEORIA ELO VS PRATICA (N=10)
Coefficiente Teorico Elo (Formula di Arpad Elo): 0.00576
Coefficiente Calcolato dal Modello (rating_diff): 0.00382
Intercetta (Vantaggio Innato del Bianco):         0.07565
